# Indian Sign Language Translation & Recognition — Baseline v1
**Interactive Google Colab Experimentation & Model Evaluation Notebook**

This notebook provides reproducible environment setup, dataset inspection, MediaPipe landmark feature extraction, normalization verification, and model evaluation for the **ISL Fingerspelling (A-Z, 1-9)** production model.

## 1. Setup & Overview
- **Task**: ISL Fingerspelling Character Recognition
- **Input**: Hand Landmarks (63D normalized coordinates)
- **Classes**: 35 classes (`1`–`9`, `A`–`Z`)
- **Model**: Deep Multi-Layer Perceptron (Scikit-Learn)
- **Test Accuracy**: 99.36%

## 2. Clone Repository

In [ ]:
# Clone the official team repository and checkout baseline-v1-team
!git clone https://github.com/yash249114/Indian-Sign-Language-Translation.git
%cd Indian-Sign-Language-Translation
!git checkout baseline-v1-team

## 3. Install Dependencies

In [ ]:
# Install core machine learning and vision dependencies
!pip install --quiet mediapipe opencv-python scikit-learn numpy joblib requests python-dotenv torch pydantic fastapi uvicorn matplotlib

## 4. Inspect Dataset Structure

In [ ]:
import json
import numpy as np

# Load official label map
with open('data/processed/landmarks/label_map.json', 'r') as f:
    label_map = json.load(f)

classes = label_map.get('classes', [])
print(f'Total Fingerspelling Classes: {len(classes)}')
print('Classes:', classes)

# Inspect processed landmark splits
test_data = np.load('data/processed/landmarks/test.npz')
print('Test split keys:', list(test_data.keys()))
print('X_test shape:', test_data['X'].shape)
print('y_test shape:', test_data['y'].shape)

## 5. Load Trained Production Model

In [ ]:
import joblib

model_path = 'models/trained/sign_classifier/mlp_classifier.joblib'
model = joblib.load(model_path)
print('Model loaded successfully:', type(model))
print('Number of input features expected:', model.n_features_in_)
print('Number of output classes:', model.n_classes_)

## 6. MediaPipe Feature Extraction & Normalization

In [ ]:
import cv2
import mediapipe as mp
import matplotlib.pyplot as plt
import sys, os

if '.' not in sys.path:
    sys.path.insert(0, '.')

from backend.app.normalizer import LandmarkNormalizer

normalizer = LandmarkNormalizer()
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5)

# Load sample image from repository
sample_img_path = 'data/sample_images/sample_A.jpg'
image = cv2.imread(sample_img_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(4, 4))
plt.imshow(image_rgb)
plt.title('Sample Image: Sign A')
plt.axis('off')
plt.show()

results = hands.process(image_rgb)
if results.multi_hand_landmarks:
    landmarks = results.multi_hand_landmarks[0]
    print('MediaPipe successfully detected 21 3D landmarks.')
    norm_vector = normalizer.normalize(landmarks)
    print(f'Extracted & Normalized 63D Vector Shape: {norm_vector.shape}')
    print('First 10 normalized coordinates:', np.round(norm_vector[:10], 4))
else:
    print('Hand not detected in sample image.')

## 7. Sample Prediction & Confidence

In [ ]:
if results.multi_hand_landmarks:
    prob = model.predict_proba([norm_vector])[0]
    pred_idx = np.argmax(prob)
    predicted_class = classes[pred_idx]
    confidence = prob[pred_idx]

    print('=' * 30)
    print(f'PREDICTED CLASS: {predicted_class}')
    print(f'CONFIDENCE:      {confidence:.4f} ({confidence * 100:.2f}%)')
    print('=' * 30 + '\n')

    # Display top 3 candidate classes
    top3_idx = np.argsort(prob)[::-1][:3]
    print('Top 3 Candidates:')
    for rank, idx in enumerate(top3_idx, 1):
        print(f'  {rank}. {classes[idx]} -> {prob[idx] * 100:.2f}%')

## 8. Test Split Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

X_test = test_data['X']
y_test = test_data['y']

y_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f'Test Set Accuracy: {test_acc * 100:.2f}%')
print('\nSummary Classification Report:')
print(classification_report(y_test, y_pred, target_names=classes, digits=4))

## 9. Model Retraining

In [ ]:
from sklearn.neural_network import MLPClassifier

train_data = np.load('data/processed/landmarks/train.npz')
X_train, y_train = train_data['X'], train_data['y']

print(f'Training new MLP on {len(X_train)} samples...')
new_mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    max_iter=150,
    random_state=42,
    early_stopping=True
)
new_mlp.fit(X_train, y_train)
new_acc = new_mlp.score(X_test, y_test)
print(f'Retrained Model Test Accuracy: {new_acc * 100:.2f}%')

## 10. Future Temporal Model Work
In Phase 2, temporal sequence models (Bi-GRU & TCN) will process continuous sequences of 126D vectors across 30 frames ($T=30$) to classify full ISL dynamic words.

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available in Colab:', torch.cuda.is_available())

# Verify baseline temporal metadata
try:
    with open('models/word_temporal_bigru.json', 'r') as f:
        meta = json.load(f)
    print('\nPhase 2 Temporal Model Target Metadata:')
    print(json.dumps(meta, indent=2))
except Exception as err:
    print('Metadata note:', err)